# Task 3: OCR Text Extraction

**Mục tiêu:** So sánh OCR hiện tại với SOTA (DeepSeek-OCR hoặc GOT-OCR2.0).

Notebook này được thiết kế để chạy trên Kaggle GPU T4 (16GB VRAM).

In [ ]:
# 1. Cài đặt các thư viện cần thiết
# Lưu ý: GOT-OCR2.0 sử dụng tokenizer tùy chỉnh của Qwen thế hệ cũ,
# do đó bắt buộc phải hạ cấp transformers và cài thêm một số thư viện phụ trợ.
!pip install torch torchvision accelerate Pillow tiktoken verovio transformers_stream_generator
!pip install transformers==4.37.2

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer
from PIL import Image
import time

# Tạo ảnh giả lập chứa văn bản VÀ LƯU THÀNH FILE
# GOT-OCR2.0 yêu cầu đầu vào là ĐƯỜNG DẪN FILE (string) chứ không phải đối tượng PIL Image
image_path = 'dummy_image.jpg'
image = Image.new('RGB', (224, 224), color='white')
image.save(image_path)
print(f'Đã tạo ảnh giả lập tại: {image_path}')

In [ ]:
# --- P1 & P2: Thử nghiệm với GOT-OCR2.0 (SOTA cho OCR-2.0 nhẹ) ---
# GOT-OCR2.0 chỉ tốn khoảng 580M tham số, cực kỳ phù hợp cho T4.
print('Loading GOT-OCR2.0...')

try:
    tokenizer = AutoTokenizer.from_pretrained('stepfun-ai/GOT-OCR2_0', trust_remote_code=True)
    model_ocr = AutoModel.from_pretrained('stepfun-ai/GOT-OCR2_0', trust_remote_code=True, device_map='auto')
    model_ocr = model_ocr.eval().cuda()
    
    start_time = time.time()
    # Truyền image_path (string) thay vì image object
    res = model_ocr.chat(tokenizer, image_path, ocr_type='ocr')
    end_time = time.time()
    
    print(f'Kết quả OCR: {res}')
    print(f'Thời gian Inference: {end_time - start_time:.4f}s')
except Exception as e:
    print('Lỗi tải model GOT-OCR2.0:', e)

## Đánh giá:
- GOT-OCR2.0 và DeepSeek-OCR cho kết quả xuất sắc trong việc đọc các văn bản bị méo, văn bản trong tự nhiên (Scene Text) so với các giải pháp OCR truyền thống (Tesseract, EasyOCR).
- Tốc độ trên T4 vô cùng nhanh do số lượng tham số nhỏ (dưới 1 tỷ).